In [1]:
# Importando as bibliotecas
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy import stats
import os
print("Bibliotecas importadas:")
print(f" geopandas {gpd.__version__}")
print(f" pandas {pd.__version__}")

Bibliotecas importadas:
 geopandas 1.2.0
 pandas 3.1.1


In [2]:
#Configurações iniciais
CODIGO_MUNICIPIO = '2112209'
AP_EXCLUIR = '2112209008'  # Área de Ponderação rural a ser excluída
# Diretórios
caminho_microdados = r'C:\Users\Waxl_Sampaio\Documents\Artigo_previa\microdados_censo2022'
caminho_shapefiles = r'C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds'
print("✅ Configurações definidas")
print(f"Município: Timon-MA ({CODIGO_MUNICIPIO})")
print(f"AP excluída (rural): {AP_EXCLUIR}")
print(f"Escola excluída (integração escolar): 21164495")

✅ Configurações definidas
Município: Timon-MA (2112209)
AP excluída (rural): 2112209008
Escola excluída (integração escolar): 21164495


In [4]:
# Estruturação do diretório de armazenamento dos dados
# Arquivos de microdados
arquivo_pessoas = os.path.join(caminho_microdados, 'Pessoas_21_controlado.txt')
arquivo_familia = os.path.join(caminho_microdados, 'Familia_21_controlado.txt')
arquivo_domicilios = os.path.join(caminho_microdados, 'Domicilios_21_controlado.txt')
# Shapefile das Áreas de Ponderação
caminho_shapefile = os.path.join(caminho_shapefiles, 'apond_timon.shp')
# Arquivos de saída
caminho_saida_gpkg = os.path.join(caminho_shapefiles, 'Timon_Apond_IVE_4D_Final.gpkg')
caminho_saida_csv = os.path.join(caminho_shapefiles, 'IVE_Timon_Apond_4D_Validacao.csv')
# Verificar existência
for nome, caminho in [
    ("Microdados Pessoas", arquivo_pessoas),
    ("Microdados Família", arquivo_familia),
    ("Microdados Domicílios", arquivo_domicilios),
    ("Shapefile APs", caminho_shapefile)
]:
    if os.path.exists(caminho):
        print(f"  {nome}: encontrado")
    else:
        print(f"  {nome}: NÃO ENCONTRADO em {caminho}")

print(f" Diretório de microdados: {caminho_microdados}")
print(f" Diretório de shapefiles: {caminho_shapefiles}")
print(f" Município: Timon-MA ({CODIGO_MUNICIPIO})")
print(f" AP excluída: {AP_EXCLUIR} (predominantemente rural)")
print(f" Total esperado de APs urbanas: 9")

  Microdados Pessoas: encontrado
  Microdados Família: encontrado
  Microdados Domicílios: encontrado
  Shapefile APs: encontrado
 Diretório de microdados: C:\Users\Waxl_Sampaio\Documents\Artigo_previa\microdados_censo2022
 Diretório de shapefiles: C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds
 Município: Timon-MA (2112209)
 AP excluída: 2112209008 (predominantemente rural)
 Total esperado de APs urbanas: 9


In [5]:
#Configuração das colunas/index Python
#Pessoas
colspecs_pess = [
    (29, 36),   # P0080 - Município (7 dígitos)
    (36, 46),   # P0090 - Área de Ponderação (10 dígitos)
    (56, 72),   # P0111 - Peso Amostral (3 int + 13 dec)
    (80, 81),   # P0140 - Situação do domicílio (1=Urbana, 2=Rural)
    (226, 227)  # P0780 - Frequência escolar adequada (1=Adequada, 2=Inadequada)
]
nomes_pess = ['P0080', 'P0090', 'P0111', 'P0140', 'P0780']

# Familia
colspecs_fami = [
    (29, 36),   # F0080 - Município
    (36, 46),   # F0090 - Área de Ponderação
    (58, 74),   # F0111 - Peso Amostral
    (78, 79),   # F0140 - Situação do domicílio
    (89, 90)    # F0190 - Nível de instrução do responsável
]
nomes_fami = ['F0080', 'F0090', 'F0111', 'F0140', 'F0190']

# Domicilio
colspecs_domi = [
    (29, 36),   # D0080 - Município
    (36, 46),   # D0090 - Área de Ponderação
    (54, 70),   # D0111 - Peso Amostral
    (74, 75),   # D0140 - Situação do domicílio
    (108, 109), # D0330 - Acesso à internet (1=Sim, 2=Não)
    (119, 130)  # D0360 - Rendimento domiciliar per capita (9 int + 2 dec)
]
nomes_domi = ['D0080', 'D0090', 'D0111', 'D0140', 'D0330', 'D0360']

print("Layouts definidos:")
print("PESSOAS: P0080, P0090, P0111, P0140, P0780")
print("FAMÍLIA: F0080, F0090, F0111, F0140, F0190")
print("DOMICÍLIOS: D0080, D0090, D0111, D0140, D0330, D0360")

Layouts definidos:
PESSOAS: P0080, P0090, P0111, P0140, P0780
FAMÍLIA: F0080, F0090, F0111, F0140, F0190
DOMICÍLIOS: D0080, D0090, D0111, D0140, D0330, D0360


In [7]:
#Leitura dos dados
#Pessoas
df_pess = pd.read_fwf(arquivo_pessoas, colspecs=colspecs_pess, names=nomes_pess, dtype=str)
print(f"Total bruto: {len(df_pess):,} registros")
# Filtros: Timon + Urbano
df_pess = df_pess[
    (df_pess['P0080'] == CODIGO_MUNICIPIO) & 
    (df_pess['P0140'] == '1') &
    (df_pess['P0090'] != AP_EXCLUIR) &
    (df_pess['P0780'].isin(['1', '2']))
].copy()
df_pess['P0111'] = df_pess['P0111'].astype(float)
print(f"Após filtros (excluindo AP {AP_EXCLUIR}): {len(df_pess):,} registros")
print(f"Frequência adequada (1): {(df_pess['P0780'] == '1').sum():,}")
print(f"Frequência inadequada (2): {(df_pess['P0780'] == '2').sum():,}")

Total bruto: 797,135 registros
Após filtros (excluindo AP 2112209008): 311 registros
Frequência adequada (1): 216
Frequência inadequada (2): 95


In [8]:
#Familia
df_fami = pd.read_fwf(arquivo_familia, colspecs=colspecs_fami, names=nomes_fami, dtype=str)
print(f"Total bruto: {len(df_fami):,} registros")
# Filtros: Timon + Urbano
df_fami = df_fami[
    (df_fami['F0080'] == CODIGO_MUNICIPIO) & 
    (df_fami['F0140'] == '1') &
    (df_fami['F0090'] != AP_EXCLUIR)
].copy()
df_fami['F0111'] = df_fami['F0111'].astype(float)
print(f"Após filtros (excluindo AP {AP_EXCLUIR}): {len(df_fami):,} registros")

Total bruto: 228,292 registros
Após filtros (excluindo AP 2112209008): 4,874 registros


In [9]:
#Domicilio
df_domi = pd.read_fwf(arquivo_domicilios, colspecs=colspecs_domi, names=nomes_domi, dtype=str)
print(f"Total bruto: {len(df_domi):,} registros")
# Filtros: Timon + Urbano
df_domi = df_domi[
    (df_domi['D0080'] == CODIGO_MUNICIPIO) & 
    (df_domi['D0140'] == '1') &
    (df_domi['D0090'] != AP_EXCLUIR)
].copy()
df_domi['D0111'] = df_domi['D0111'].astype(float)
df_domi['D0360'] = df_domi['D0360'].astype(float) / 100  # Correção das 2 casas decimais
print(f"Após filtros (excluindo AP {AP_EXCLUIR}): {len(df_domi):,} registros")
print(f"Renda per capita média: R$ {df_domi['D0360'].mean():,.2f}")
print(f"Internet:")
print(f" Com (1): {(df_domi['D0330'] == '1').sum():,}")
print(f" Sem (2): {(df_domi['D0330'] == '2').sum():,}")

Total bruto: 244,256 registros
Após filtros (excluindo AP 2112209008): 5,102 registros
Renda per capita média: R$ 1,072.41
Internet:
 Com (1): 4,287
 Sem (2): 774


In [10]:
#Agregação - Proporção Fora da Escola
agg_pess = df_pess.groupby('P0090').apply(
    lambda x: pd.Series({
        'TOTAL_PESSOAS': x['P0111'].sum(),
        'FORA_ESCOLA': x.loc[x['P0780'] == '2', 'P0111'].sum()
    })
).reset_index()
agg_pess['PROP_FORA_ESCOLA'] = agg_pess['FORA_ESCOLA'] / agg_pess['TOTAL_PESSOAS']
print(f"{len(agg_pess)} APs")

9 APs


In [11]:
#Agregação - Baixa Instrução do Responsável
agg_fami = df_fami.groupby('F0090').apply(
    lambda x: pd.Series({
        'TOTAL_FAM': x['F0111'].sum(),
        'BAIXA_INSTR': x.loc[x['F0190'] == '1', 'F0111'].sum()
    })
).reset_index()
agg_fami['PROP_BAIXA_INSTR'] = agg_fami['BAIXA_INSTR'] / agg_fami['TOTAL_FAM']
print(f"{len(agg_fami)} APs")

9 APs


In [12]:
#Agregação - Renda e Internet
agg_domi = df_domi.groupby('D0090').apply(
    lambda x: pd.Series({
        'RENDA_MEDIA': np.average(x['D0360'], weights=x['D0111']),
        'TOTAL_DOM': x['D0111'].sum(),
        'SEM_INTERNET': x.loc[x['D0330'] == '2', 'D0111'].sum()
    })
).reset_index()
agg_domi['PROP_SEM_INTERNET'] = agg_domi['SEM_INTERNET'] / agg_domi['TOTAL_DOM']
print(f"{len(agg_domi)} APs")

9 APs


In [13]:
#Junção das Tabelas
# Padronização de chave
agg_pess = agg_pess.rename(columns={'P0090': 'AREA_POND'})
agg_fami = agg_fami.rename(columns={'F0090': 'AREA_POND'})
agg_domi = agg_domi.rename(columns={'D0090': 'AREA_POND'})
# Outer join sequencial
df_ive = agg_pess[['AREA_POND', 'PROP_FORA_ESCOLA']].copy()
df_ive = df_ive.merge(agg_fami[['AREA_POND', 'PROP_BAIXA_INSTR']], on='AREA_POND', how='outer')
df_ive = df_ive.merge(agg_domi[['AREA_POND', 'RENDA_MEDIA', 'PROP_SEM_INTERNET']], on='AREA_POND', how='outer')
print(f" {len(df_ive)} APs na base unificada")

 9 APs na base unificada


In [14]:
#Tratamento de NaNs
for col in ['PROP_FORA_ESCOLA', 'PROP_BAIXA_INSTR', 'RENDA_MEDIA', 'PROP_SEM_INTERNET']:
    nulos = df_ive[col].isna().sum()
    if nulos > 0:
        df_ive[col] = df_ive[col].fillna(df_ive[col].median())
        print(f"{col}: {nulos} preenchido(s)")
    else:
        print(f"{col}: OK ✅")

PROP_FORA_ESCOLA: OK ✅
PROP_BAIXA_INSTR: OK ✅
RENDA_MEDIA: 4 preenchido(s)
PROP_SEM_INTERNET: OK ✅


In [15]:
#Cálculo dos Z-Scores
# Variáveis DIRETAS (maior = maior vulnerabilidade)
df_ive['Z_FORA_ESCOLA'] = stats.zscore(df_ive['PROP_FORA_ESCOLA'])
df_ive['Z_BAIXA_INSTR'] = stats.zscore(df_ive['PROP_BAIXA_INSTR'])
df_ive['Z_SEM_INTERNET'] = stats.zscore(df_ive['PROP_SEM_INTERNET'])
# Variável INVERSA (maior renda = MENOR vulnerabilidade)
df_ive['Z_RENDA'] = -stats.zscore(df_ive['RENDA_MEDIA'])
print("4 Z-scores calculados")

4 Z-scores calculados


In [16]:
#Cálculo do IVE
df_ive['IVE_FINAL_4D'] = (
    df_ive['Z_FORA_ESCOLA'] + 
    df_ive['Z_BAIXA_INSTR'] + 
    df_ive['Z_SEM_INTERNET'] + 
    df_ive['Z_RENDA']
) / 4
print("IVE")
print(f"Ranking de Vulnerabilidade ({len(df_ive)} APs urbanas):")
ranking = df_ive[['AREA_POND', 'IVE_FINAL_4D']].sort_values('IVE_FINAL_4D', ascending=False).reset_index(drop=True)
ranking.index = ranking.index + 1
print(ranking)

IVE
Ranking de Vulnerabilidade (9 APs urbanas):
    AREA_POND  IVE_FINAL_4D
1  2112209004      0.515645
2  2112209009      0.457124
3  2112209002      0.333960
4  2112209001      0.196863
5  2112209006      0.065869
6  2112209007      0.046839
7  2112209003      0.003883
8  2112209005     -0.638157
9  2112209010     -0.982025


In [17]:
#Junção Espacial
gdf_apond = gpd.read_file(caminho_shapefile)
print(f"{len(gdf_apond)} feições carregadas")
# Excluir AP rural do shapefile (consistência)
gdf_apond = gdf_apond[gdf_apond['cd_apond'] != AP_EXCLUIR].copy()
print(f"AP {AP_EXCLUIR} excluída. Restam {len(gdf_apond)} feições.")

10 feições carregadas
AP 2112209008 excluída. Restam 9 feições.


In [18]:
#Mesclagem
gdf_final = gdf_apond.merge(df_ive, left_on='cd_apond', right_on='AREA_POND', how='left')
if 'AREA_POND' in gdf_final.columns:
    gdf_final = gdf_final.drop(columns=['AREA_POND'])
print(f"{len(gdf_final)} feições finais")

9 feições finais


In [19]:
#Exportação GeoPackage
gdf_final.to_file(caminho_saida_gpkg, driver='GPKG', encoding='utf-8')
print(f"GeoPackage salvo: {caminho_saida_gpkg}")

GeoPackage salvo: C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds\Timon_Apond_IVE_4D_Final.gpkg


In [20]:
#Exportação CSV
colunas_csv = [
    'cd_apond', 'PROP_FORA_ESCOLA', 'PROP_BAIXA_INSTR', 
    'RENDA_MEDIA', 'PROP_SEM_INTERNET', 'IVE_FINAL_4D'
]
gdf_final[colunas_csv].to_csv(
    caminho_saida_csv, 
    index=False, 
    sep=';', 
    decimal=',',
    encoding='utf-8'
)
print(f"CSV salvo: {caminho_saida_csv}")

CSV salvo: C:\Users\Waxl_Sampaio\Documents\Artigo_previa\Escolas_Aponds\IVE_Timon_Apond_4D_Validacao.csv
